## 08 Abschlussbericht: Erkenntnisse, Kritische Reflexion & Ausblick

## 1. Erkenntniszusammenfassung mit Kontextbezug

In diesem Projekt haben wir verschiedene Modellansätze untersucht, um die logarithmierte Like-Anzahl (`likes_log`) von Instagram-Posts vorherzusagen. Dabei standen Metadaten (z. B. Follower-Anzahl, Kommentar-Count, Hashtag-Count, Account-Kategorie) und Bilddaten (ResNet50-Features) im Fokus.


### Überblick der Modellleistungen

| Modell                        | Datenquelle         | Test-MSE  | Test-R²  | Kurze Einordnung                                           |
|:------------------------------|:--------------------|:----------|:---------|:------------------------------------------------------------|
| **OLS-Regression**            | Metadaten           | ≈ 4.00    | ≈ 0.65   | Solide Baseline, aber lineare Annahmen limitieren Korrelationen. |
| **Random Forest (RF)**        | Metadaten           | ≈ 1.50    | ≈ 0.87   | Sehr gute Leistung, fängt nichtlineare Effekte und Interaktionen ein. |
| **CNN (Bild-Mittelwert)**     | Bild-Features       | ≈ 15.87   | ≈ –0.26  | Geringe Vorhersagekraft – Datenbasis zu klein, Aggregation zu einfach. |
| **Stacking (RF + CNN)**       | Metadaten + Bild    | ≈ 1.60    | ≈ 0.873  | Minimal besser als RF; Bildsignal wird kaum gewichtet.       |

- **Random Forest (Metadaten)** liefert die stärkste Vorhersage (R² ≈ 0.87).  
  - Die wichtigsten Prädiktoren sind `comment_count` (stärkster Einfluss), `follower_count` (Reichweite) und `account_category_Musician`.  
  - RF erklärt deutlich mehr Varianz als OLS (R² ≈ 0.65) und zeigt, dass Metadaten-Interaktionen relevant sind.

- **CNN allein (Bild-Features)** brachte in dieser Konfiguration kaum Mehrwert (R² negativ).  
  - Ursache: Nur rund 2 000 Posts mit je wenigen Bildern, einfache Feature-Aggregation (arithmetischer Mittelwert), kein Fine-Tuning.  
  - Visuelle Merkmale bleiben ohne größere Datenbasis und spezialisierte Bild-Netzwerke (z. B. Fine-Tuning von ResNet50) zu ungenau.

- **Stacking-Modell** kombiniert RF- und CNN-Vorhersagen via linearer Meta-Regressor.  
  - Meta-Koeffizienten zeigen hohes Gewicht auf RF (≈ 1.05) und nahezu Null auf CNN (≈ 0.007).  
  - Gesamtleistung (R² ≈ 0.873) verbessert sich nur marginal gegenüber RF allein (R² ≈ 0.8695), da das Bildsignal in dieser Version zu schwach ist.


###  Kontext und praktische Relevanz

- **Marketing & Influencer-Tools:**  
  - Eine valide Like-Prognose auf Basis von Metadaten allein ist in vielen beruflichen Anwendungen ausreichend (Agenturen, Brand Manager, Influencer).  
  - Vor allem Follower- und Kommentar-Zahlen sind leicht zugänglich und korrelieren stark mit dem Engagement.  
  - Ein einfaches RF-Modell kann in einem Tool integriert werden, um Content Creator:innen zu zeigen, wie viele Likes sie realistischerweise erwarten können – noch bevor der Beitrag online geht.

- **Nutzungsszenarien:**  
  1. **A/B-Tests** von Hashtag-Strategien: Verschiedene Hashtag-Kombinationen eingeben, Prognose für jeden Szenarienvergleich.  
  2. **Timing-Optimierung:** Ergänzung mit Uhrzeit-/Tageszeit-Variablen (in späteren Modellen) liefert Hinweise, wann ein Post am wahrscheinlichsten hohe Likes erzielt.  
  3. **Account-Benchmarking:** Prognostizierte Likes in Relation zum Account-Durchschnitt setzen – damit lassen sich realistische Zielvorgaben für größere Kampagnen ableiten.


## Kritische Reflexion -> Limitationen, potenzielle Fehlerquellen und offene Fragen

### Datenbasis und Generalisierbarkeit

- **Stichprobengröße:**  
  - Nur 1 968 Posts, meist aus wenigen Accounts (z. B. „Digital creator“). Kleine, homogene Stichprobe begrenzt Generalisierbarkeit.  
  - Insbesondere für CNN-Modelle sind Millionen von Bildern üblich, um robuste Feature-Detektoren zu trainieren. Unsere 2 000 Posts sind zu wenig, daher negative R²-Werte bei reinen CNN-Vorhersagen.

- **Kontextspezifität:**  
  - Merkmale wie Account-Kategorie oder typische Hashtag-Strategien können je nach Nische (Mode, Food, Fitness) stark variieren.  
  - Ein Modell, das nur „Digital creator“ beinhaltet, wird Schwierigkeiten haben, in völlig anderen Nischen valide zu predizieren.

### Modellannahmen und -begrenzungen

- **OLS-Regression (Linearkonstrukt):**  
  - Macht strikte Annahmen (Lineari­tät, Normalverteilung der Residuen, Homoskedastizität). Bei heterogenen Social-Media-Daten selten vollständig erfüllt.  
  - Residuen plots zeigen starke Abweichungen, insbesondere bei sehr niedrigen oder sehr hohen Like-Zahlen.

- **Random Forest (Black-Box-​Charakter):**  
  - Auch wenn RF nichtlineare Beziehungen abbildet, bleibt Interpretierbarkeit eingeschränkt. Feature-Importances geben nur globales Ranking, keine lokalen Zusammenhänge.  
  - Partial-Dependence-Plots (PDPs) oder SHAP-Werte wären nötig, um lokales Verhalten besser zu verstehen.

- **CNN (Bild-Features):**  
  - ResNet50-Features sind vorkonditioniert auf ImageNet-Klassen (Alltagsobjekte), nicht spezifisch für Social-Media-Ästhetik.  
  - Bildaggregationsmethode (arithmetischer Mittelwert über alle Carousel-Bilder) vernachlässigt, dass oft nur das erste Bild das Engagement bestimmt.  
  - Kein Fine-Tuning: Wir haben das ResNet50-Backbone *eingefroren*. Ein feineres Training auf Instagram-spezifischen Stilen könnte die Bildleistung substantiell verbessern.

- **Stacking (Meta-Regressor):**  
  - Nutzt lineare Regression als Meta-Modell. Das ist simpel, aber unterstellt Linearkombination beider Vorhersagen.  
  - Wenn die Einzelmodelle stark korreliert sind (RF und CNN-Hochdimensionalität), kann Stacking keinen großen Mehrwert mehr erzielen.

### Feature Engineering

- **Metadaten-Features:**  
  - Wichtige Merkmale wie `comment_count` und `follower_count` sind zwar stark prädiktiv, aber wir haben sie nur in Rohform genutzt.  
  - Zusätzliche Merkmale (z. B. Caption-Sentiment, Uhrzeit-Embeddings) wurden nicht implementiert, könnten aber weitere Varianz erklären.  

- **Bild-Features:**  
  - Neben reinen Pixel-Features könnten z. B. *Farbanalysen*, *Objekterkennung (Gesichter, Markenlogos)* oder *Ästhetikmetriken* wertvolle Informationen liefern.  
  - Unser Einfachmodell ignoriert gesondert Bildkomposition, Farbpalette, Textüberlagerung etc.

### Modell-Drift & Wartung

- **Statische Modelle vs. dynamische Plattform:**  
  - Instagram-Algorithmen ändern sich regelmäßig (Feed-Ranking, Reels-Vorzug). Ein Modell, das heute gute Performance zeigt, kann über Nacht inadäquat sein.  
  - Eine kontinuierliche Modellüberwachung (Monitoring der Metriken, regelmäßiges Retraining) ist essenziell, bleibt in diesem Projekt jedoch unberücksichtigt.

- **Trainings-/Test-Split**  
  - Wir haben einen zufälligen 60/40-Split genutzt, ohne zeitliche Reihenfolge zu beachten. In der Praxis sind Posts zeitabhängig: Trends, Saisonaleffekte (Feiertage, Events) könnten zu Leaks im Modell führen, wenn historische Posts zufällig im Testset landen.

### Ethische und Datenschutz-Aspekte

- **Personenbezogene Inhalte:**  
  - Hochgeladene Bilder können Gesichter oder private Informationen enthalten. Ein Webservice müsste sicherstellen, dass Bilder nach der Inferenz unverzüglich gelöscht oder verschlüsselt archiviert werden.  
  - DSGVO-Konformität bei Nutzung von Metadaten (z. B. `is_verified`) muss gewährleistet sein, insbesondere wenn personenbezogene Daten an Dritte weitergegeben werden.

- **Bias in den Daten:**  
  - Accounts mit vielen Followern und hohem Engagement sind stärker vertreten, kleinere Nischen-Accounts (unter 10 000 Follower) kaum.  
  - Das Modell könnte systematisch überschätzen, wie viele Likes ein kleiner Account erhält, wenn die Trainingsdaten überwiegend große Accounts abdecken.



## Ausblick

Um die Modelle in der Praxis nutzbar zu machen und die genannten Limitationen anzugehen, skizziere ich im Folgenden einen fokussierten Ausblick:

### Nächste Schritte für Modellverbesserungen

1. **Datenvergrößerung & Diversifikation**  
   - Eine Größenordnung von beispielsweise mindestens 10 000–50 000 Instagram-Posts aus unterschiedlichen Nischen sammeln 
   - Größeres Bild‐ und Metadatenspektrum: CNN kann tiefergehende Muster (z. B. Farbästhetik in Mode- vs. Food-Posts) lernen.  

2. **Feinabstimmung des Bild-Modells (Fine-Tuning)**  
   - ResNet50 nicht nur als bloßer Feature-Extraktor: Stattdessen die letzten Convolutional-Blöcke trainierbar machen („unfreeze“) und mit kleinen Lernraten auf Instagram-Daten nachtrainieren.  
   - **Transfer Learning** mit zusätzlichen Regularisierungsmaßnahmen (Dropout, Data Augmentation) um Überanpassung zu vermeiden.

3. **Erweiterte Feature Engineering bei Metadaten**  
   - **Textbasierte Merkmale:** Caption-Sentiment, Häufigkeit bestimmter Schlüsselwörter (z. B. Markennamen).  
   - **Zeitliche Effekte:** Uhrzeit (Stichwort „beste Posting-Zeit“), Wochentag, saisonale Saisons (z. B. Feiertage).  
   - **Interaktionsmetriken:** Anzahl der Views in den ersten 30 Minuten (falls vorhanden), Watchtime, Saves, Shares.  

4. **Optimierte Aggregation von Bild-Vorhersagen**  
   - Statt arithmetischem Mittel:  
     - **Max-Pooling:** Nur das Bild mit der höchsten Einzelvorhersage pro Post zählt.  
     - **Gewichtetes Mittel:** Gewichtung nach Position im Carousel (erstes Bild bekommt hohen Faktor).  
     - **Attention-Mechanismus:** Ein kleines Sequenzmodell lernt, welche Bildauswahl wichtiger ist.  

5. **Erweiterung des Meta-Regressors**  
   - Anstelle einer rein linearen Regression ggf. eine **LightGBM- oder XGBoost-Stacking-Ebene**, die nichtlineare Kombinationen von RF- und CNN-Vorhersagen lernt.  
   - **Kreuzvalidierung** auf Meta-Ebene, um Overfitting zwischen Einzel- und Meta-Level zu vermeiden.


###  Web-Interface für Like-Prognosen

Für eine praxisnahe Anwendung schlage ich den Aufbau einer schlanken Webplattform vor, in der Content Creator:innen und Agenturen sofort interaktiv und visuell ihre Posting-Strategien evaluieren können.

####  Kernfunktionalitäten

1. **Upload & Metadaten-Eingabe**  
   - **Bild-Upload:** Mehrere Bilder pro Post (Carousel) hochladen. Vorschau im Browser anzeigen.  
   - **Metadaten-Formular:**  
     - Follower-Zahl (numerische Eingabe)  
     - Kommentar-Schätzung (z. B. in Prozent der Follower)  
     - Hashtag-Count (Dropdown oder Schieberegler)  
     - Account-Kategorie (Auswahl­feld)  
     - Verifiziert (Checkbox), Geschäftsaccount (Checkbox)

2. **Automatische Inferenz & Ergebnisvisualisierung**  
   - Nach Klick auf „Prognose erstellen“ sendet das Frontend ein JSON-Payload an das Backend, das in Echtzeit folgende Schritte ausführt:  
     1. **Metadaten -> Numerical & OHE -> RF-Vorhersage**  
     2. **Bilder -> ResNet50 (Fine-Tuned) -> Aggregation -> CNN-Vorhersage**  
     3. **Stacking-Regressor -> finale `likes_log`**  
     4. Rücktransformation: Aus `likes_log` wird `likes_pred = round(exp(likes_log))`  
   - Das Frontend zeigt:  
     - Die **prognostizierte absolute Like-Anzahl** (z. B. 12 345 Likes)  
     - Den **log-transformierten Wert** (z. B. 9.42 log-Likes)  
     - **Balkendiagramm**, das Prognose vs. Account-Durchschnitt vergleicht.  
     - Optional: **Intervalle** („wahrscheinlicher Bereich: 11 000–13 000 Likes“) basierend auf Residuen-Streuung.

3. **Vergleich verschiedener Varianten (A/B-Test-Modus)**  
   - Nutzer:innen können mehrere Szenarien definieren (Variante A: 5 Hashtags, Variante B: 10 Hashtags; Variante C: unterschiedlicher Caption-Text).  
   - Ergebnisse in einem gestapelten Balkendiagramm nebeneinander anzeigen, um zu entscheiden, welche Strategie höheres Engagement verspricht.

4. **Automatische Empfehlungen (Rule-Based)**  
   - Basierend auf **RF-Feature Importances** und partielle Abhängigkeiten:  
     - „Erhöhe Hashtag-Anzahl von aktuell 3 auf mindestens 7, um erwartete Likes um etwa 10 % zu steigern.“  
     - „Dein Kommentar-Engagement liegt unter dem Top-Quartil, erwäge mehr direkte Call-to-Action im Text.“  
     - „Wenn Account nicht verifiziert ist, sinkt der Mittelwert der prognostizierten Likes um ca. 15 %.“  
   - Diese Empfehlungen erscheinen unterhalb des Prognose-Outputs als Textsummarien.

5. **Historische Datenbank & Modell-Aktualisierung**  
   - Jeder tatsächlich gepostete Beitrag mit finaler Like-Zahl wird (manuell oder automatisiert) in eine Datenbank übernommen.  
   - Ein geplanter Hintergrundjob (z. B. wöchentlich) übernimmt:  
     1. Einlesen neuer Real-World-Daten  
     2. Neu­training des RF-Modells (ggf. mit erweiterten Features)  
     3. Optional: Neu­fine­tuning des CNN (wenn genügend neue Bilder vorliegen)  
     4. Versionierung und Deployment des aktualisierten Modells  

6. **Externe API-Schnittstelle**  
   - **REST-Endpoint** 
   - **Response:**  
   - **Dokumentation** über Swagger/OpenAPI, um Drittanbieter-Tools (CMS, Social Media Scheduler) zu integrieren.

#### Datenschutz & Sicherheit

- **Bildlöschung:**  
  - Nach Inferenz müssen alle Bilddateien automatisch gelöscht werden (z. B. nach 24 Stunden).  
  - Sensible Daten wie `is_verified` sollten pseudonymisiert oder verschlüsselt gespeichert werden.

- **Authentifizierung & Autorisierung:**  
  - **OAuth2 / JWT**: Token-basierte Authentifizierung, um API-Endpunkte vor unbefugtem Zugriff zu schützen.  
  - **HTTPS** (TLS) zwingend für alle Datenverbindungen.

- **Model-Versionierung & Monitoring:**  
  - Jede Modellversion (RF / CNN / Stacking) erhält eindeutigen Tag (Datum + Versionsnummer).  
  - Ein **Monitoring-Dashboard** (z. B. Grafana) protokolliert:  
    - Modell-Latenzen (Inference Time)  
    - Qualitätskennzahlen (Test-R², MSE) über die letzten Retrains  
    - Nutzungsstatistiken (Anzahl der Anfragen pro Tag)


##  Gesamtfazit

- **Metadaten-Modelle (RF) sind in der aktuellen Stichprobe das stärkste Fundament** für Like-Vorhersagen. Sie erreichen R² ≈ 0.87 und erklären damit praktisch den größten Teil der Varianz.  
- **Bilddaten (CNN) können bei kleinen Datenmengen kaum robuste Patterns extrahieren.** In dieser Ausbaustufe führte das CNN zu negativen R²-Werten.  
- **Stacking erhöht den Vorhersagewert nur marginal**, da das Bildsignal zu schwach ist.  
- **Einfaches Web-Tool, das RF nutzt, reicht in der Praxis oft aus.** Ein Limit der Arbeit war die begrenzte Datenbasis; große, heterogene Instagram-Datensätze würden Bildmodelle deutlich aufwerten.  
- **Für reale Anwendungen** empfiehlt es sich, mit RF zu starten, kontinuierlich neue Daten zu sammeln, Modell-Drift zu überwachen und Bild-Modelle erst hinzuziehen, wenn ein ausreichend großer, qualitativ hochwertiger Bilddatensatz vorhanden ist.
